
# LAB 5 — Letter Recognition Using ANNs


- MNIST and Letter Recognition datasets
- ANN trained from scratch with NumPy
- Forward propagation, backpropagation and gradient descent
- Random, Xavier and Kaiming/He weight initialization
- ReLU, Leaky ReLU, Tanh and Sigmoid activations
- Dropout regularization
- 2-, 5- and 10-hidden-layer architectures
- Training/validation accuracy curves
- Confusion matrices
- Feature histograms and correlation heatmap for Letter Recognition




In [ ]:


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

np.random.seed(42)

plt.rcParams["figure.figsize"] = (8, 5)


## 2. Activation functions

In [ ]:

def relu(z):
    return np.maximum(0, z)

def relu_derivative(z):
    return (z > 0).astype(float)

def leaky_relu(z, alpha=0.01):
    return np.where(z > 0, z, alpha * z)

def leaky_relu_derivative(z, alpha=0.01):
    return np.where(z > 0, 1.0, alpha)

def tanh(z):
    return np.tanh(z)

def tanh_derivative(z):
    a = np.tanh(z)
    return 1 - a**2

def sigmoid(z):
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))

def sigmoid_derivative(z):
    a = sigmoid(z)
    return a * (1 - a)

def softmax(z):
    z = z - np.max(z, axis=1, keepdims=True)
    exp_z = np.exp(z)
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)

ACTIVATIONS = {
    "relu": (relu, relu_derivative),
    "leaky_relu": (leaky_relu, leaky_relu_derivative),
    "tanh": (tanh, tanh_derivative),
    "sigmoid": (sigmoid, sigmoid_derivative),
}

# Plot activation functions and derivatives
x = np.linspace(-5, 5, 400)

plt.figure(figsize=(10, 6))
for name, (fn, _) in ACTIVATIONS.items():
    plt.plot(x, fn(x), label=name)

plt.xlabel("x")
plt.ylabel("Activation")
plt.title("Activation Functions")
plt.legend()
plt.grid()
plt.show()

plt.figure(figsize=(10, 6))
for name, (_, derivative) in ACTIVATIONS.items():
    plt.plot(x, derivative(x), label=f"{name} derivative")

plt.xlabel("x")
plt.ylabel("Derivative")
plt.title("Activation Function Derivatives")
plt.legend()
plt.grid()
plt.show()


## 3. Weight initialization

In [ ]:

def initialize_weights(layer_sizes, method="random", seed=42):
    rng = np.random.default_rng(seed)
    weights = []
    biases = []

    for i in range(len(layer_sizes) - 1):
        fan_in = layer_sizes[i]
        fan_out = layer_sizes[i + 1]

        if method == "random":
            W = rng.normal(0, 0.01, (fan_in, fan_out))

        elif method == "xavier":
            # Xavier/Glorot normal initialization
            std = np.sqrt(2.0 / (fan_in + fan_out))
            W = rng.normal(0, std, (fan_in, fan_out))

        elif method in ("kaiming", "he"):
            # He/Kaiming normal initialization
            std = np.sqrt(2.0 / fan_in)
            W = rng.normal(0, std, (fan_in, fan_out))

        else:
            raise ValueError("method must be random, xavier, or kaiming")

        b = np.zeros((1, fan_out))

        weights.append(W)
        biases.append(b)

    return weights, biases


## 4. ANN from scratch

In [ ]:

def one_hot_encode(y, n_classes=None):
    y = np.asarray(y, dtype=int)
    if n_classes is None:
        n_classes = int(y.max()) + 1

    Y = np.zeros((len(y), n_classes))
    Y[np.arange(len(y)), y] = 1
    return Y


def forward_pass(X, weights, biases, activation_name="relu",
                 training=True, dropout_rate=0.0, rng=None):
    activation, _ = ACTIVATIONS[activation_name]

    A = X
    activations = [A]
    pre_activations = []
    dropout_masks = []

    for layer in range(len(weights) - 1):
        Z = A @ weights[layer] + biases[layer]
        A = activation(Z)

        if training and dropout_rate > 0:
            if rng is None:
                rng = np.random.default_rng()

            mask = (rng.random(A.shape) >= dropout_rate)
            A = A * mask / (1 - dropout_rate)
        else:
            mask = None

        pre_activations.append(Z)
        activations.append(A)
        dropout_masks.append(mask)

    # Output layer: softmax
    Z = A @ weights[-1] + biases[-1]
    output = softmax(Z)

    pre_activations.append(Z)
    activations.append(output)
    dropout_masks.append(None)

    return output, activations, pre_activations, dropout_masks


def cross_entropy(Y, P):
    eps = 1e-12
    P = np.clip(P, eps, 1 - eps)
    return -np.mean(np.sum(Y * np.log(P), axis=1))


def backward_pass(X, Y, weights, activations, pre_activations,
                  dropout_masks, activation_name="relu"):
    _, activation_derivative = ACTIVATIONS[activation_name]

    m = X.shape[0]
    dW = [None] * len(weights)
    db = [None] * len(weights)

    # Softmax + cross-entropy derivative
    dZ = activations[-1] - Y

    for layer in range(len(weights) - 1, -1, -1):
        dW[layer] = (activations[layer].T @ dZ) / m
        db[layer] = np.sum(dZ, axis=0, keepdims=True) / m

        if layer > 0:
            dA_prev = dZ @ weights[layer].T

            mask = dropout_masks[layer - 1]
            if mask is not None:
                # Inverted dropout was used during forward propagation.
                dA_prev = dA_prev * mask / (1 - np.mean(mask == 0))

            dZ = dA_prev * activation_derivative(pre_activations[layer - 1])

    return dW, db


def predict(X, weights, biases, activation_name="relu"):
    P, _, _, _ = forward_pass(
        X, weights, biases,
        activation_name=activation_name,
        training=False,
        dropout_rate=0
    )
    return np.argmax(P, axis=1)


def train_ann(
    X_train, y_train,
    X_val, y_val,
    hidden_layers=(128, 64),
    activation_name="relu",
    init_method="random",
    epochs=50,
    learning_rate=0.1,
    batch_size=128,
    dropout_rate=0.0,
    seed=42,
    verbose=True
):
    n_features = X_train.shape[1]
    n_classes = int(max(y_train.max(), y_val.max())) + 1

    layer_sizes = [n_features] + list(hidden_layers) + [n_classes]

    weights, biases = initialize_weights(
        layer_sizes,
        method=init_method,
        seed=seed
    )

    Y_train = one_hot_encode(y_train, n_classes)
    Y_val = one_hot_encode(y_val, n_classes)

    rng = np.random.default_rng(seed)

    history = {
        "train_loss": [],
        "val_loss": [],
        "train_accuracy": [],
        "val_accuracy": []
    }

    for epoch in range(epochs):
        indices = rng.permutation(len(X_train))
        X_shuffled = X_train[indices]
        Y_shuffled = Y_train[indices]

        for start in range(0, len(X_train), batch_size):
            end = start + batch_size

            X_batch = X_shuffled[start:end]
            Y_batch = Y_shuffled[start:end]

            P, activations, pre_activations, dropout_masks = forward_pass(
                X_batch,
                weights,
                biases,
                activation_name=activation_name,
                training=True,
                dropout_rate=dropout_rate,
                rng=rng
            )

            dW, db = backward_pass(
                X_batch,
                Y_batch,
                weights,
                activations,
                pre_activations,
                dropout_masks,
                activation_name=activation_name
            )

            for i in range(len(weights)):
                weights[i] -= learning_rate * dW[i]
                biases[i] -= learning_rate * db[i]

        # Evaluation
        train_P, _, _, _ = forward_pass(
            X_train, weights, biases,
            activation_name=activation_name,
            training=False
        )

        val_P, _, _, _ = forward_pass(
            X_val, weights, biases,
            activation_name=activation_name,
            training=False
        )

        train_loss = cross_entropy(Y_train, train_P)
        val_loss = cross_entropy(Y_val, val_P)

        train_pred = np.argmax(train_P, axis=1)
        val_pred = np.argmax(val_P, axis=1)

        train_acc = accuracy_score(y_train, train_pred)
        val_acc = accuracy_score(y_val, val_pred)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_accuracy"].append(train_acc)
        history["val_accuracy"].append(val_acc)

        if verbose and ((epoch + 1) % 5 == 0 or epoch == 0):
            print(
                f"Epoch {epoch+1:02d}/{epochs} | "
                f"Train Acc: {train_acc:.4f} | "
                f"Val Acc: {val_acc:.4f} | "
                f"Train Loss: {train_loss:.4f} | "
                f"Val Loss: {val_loss:.4f}"
            )

    return weights, biases, history


## 5. Evaluation and plotting functions

In [ ]:

def plot_history(history, title="ANN Training"):
    epochs = range(1, len(history["train_accuracy"]) + 1)

    plt.figure(figsize=(9, 5))
    plt.plot(epochs, history["train_accuracy"], label="Training Accuracy")
    plt.plot(epochs, history["val_accuracy"], label="Validation Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title(title)
    plt.legend()
    plt.grid()
    plt.show()


def show_confusion_matrix(y_true, y_pred, class_names=None, title="Confusion Matrix"):
    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(9, 7))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=class_names if class_names is not None else "auto",
        yticklabels=class_names if class_names is not None else "auto"
    )
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(title)
    plt.show()

    print(classification_report(
        y_true,
        y_pred,
        target_names=class_names if class_names is not None else None
    ))


def run_experiment(
    X_train, y_train, X_val, y_val,
    name,
    hidden_layers=(128, 64),
    activation_name="relu",
    init_method="random",
    dropout_rate=0.0,
    epochs=50,
    learning_rate=0.1,
    batch_size=128,
    seed=42
):
    print(f"\n{'='*70}")
    print(name)
    print(f"{'='*70}")

    weights, biases, history = train_ann(
        X_train, y_train,
        X_val, y_val,
        hidden_layers=hidden_layers,
        activation_name=activation_name,
        init_method=init_method,
        epochs=epochs,
        learning_rate=learning_rate,
        batch_size=batch_size,
        dropout_rate=dropout_rate,
        seed=seed
    )

    val_pred = predict(
        X_val, weights, biases,
        activation_name=activation_name
    )

    final_val_accuracy = accuracy_score(y_val, val_pred)

    print(f"Final validation accuracy: {final_val_accuracy:.4f}")

    plot_history(history, name)

    return {
        "name": name,
        "weights": weights,
        "biases": biases,
        "history": history,
        "predictions": val_pred,
        "accuracy": final_val_accuracy
    }


## 6. Load MNIST

In [ ]:

# Option A: use an existing mnist_train.csv with the first column as label.
# Expected structure: label, pixel1, pixel2, ..., pixel784

import os

MNIST_FILE = "mnist_train.csv"

if os.path.exists(MNIST_FILE):
    mnist_df = pd.read_csv(MNIST_FILE)

    # If the first column is the label
    y_mnist = mnist_df.iloc[:, 0].values.astype(int)
    X_mnist = mnist_df.iloc[:, 1:].values.astype(np.float32) / 255.0

else:
    # Fallback to Keras MNIST if TensorFlow is installed.
    try:
        from tensorflow.keras.datasets import mnist

        (X_train_raw, y_train_raw), (X_test_raw, y_test_raw) = mnist.load_data()

        X_mnist = np.concatenate([
            X_train_raw.reshape(len(X_train_raw), -1),
            X_test_raw.reshape(len(X_test_raw), -1)
        ]).astype(np.float32) / 255.0

        y_mnist = np.concatenate([y_train_raw, y_test_raw]).astype(int)

    except Exception as e:
        raise FileNotFoundError(
            "Place mnist_train.csv in the notebook folder, "
            "or install TensorFlow so the Keras MNIST fallback can be used."
        ) from e

print("MNIST shape:", X_mnist.shape)
print("Labels:", np.unique(y_mnist))


In [ ]:

# 7. MNIST train/validation split

X_mnist_train, X_mnist_val, y_mnist_train, y_mnist_val = train_test_split(
    X_mnist,
    y_mnist,
    test_size=0.20,
    random_state=42,
    stratify=y_mnist
)

print("Training:", X_mnist_train.shape)
print("Validation:", X_mnist_val.shape)


## 8. MNIST — Random, Xavier and Kaiming initialization

In [ ]:

mnist_results = {}

for init in ["random", "xavier", "kaiming"]:
    result = run_experiment(
        X_mnist_train, y_mnist_train,
        X_mnist_val, y_mnist_val,
        name=f"MNIST — {init.capitalize()} Initialization",
        hidden_layers=(128, 64),
        activation_name="relu",
        init_method=init,
        epochs=50,
        learning_rate=0.1,
        batch_size=128,
        seed=42
    )

    mnist_results[init] = result


In [ ]:

# Compare MNIST results

mnist_comparison = pd.DataFrame({
    "Initialization": list(mnist_results.keys()),
    "Validation Accuracy": [
        result["accuracy"] for result in mnist_results.values()
    ]
})

mnist_comparison


## 9. Load Letter Recognition dataset

In [ ]:

# UCI Letter Recognition dataset:
# letter-recognition.data
#
# The file normally contains 17 columns:
# letter + 16 numerical attributes.

LETTER_FILE = "letter-recognition.data"

if not os.path.exists(LETTER_FILE):
    raise FileNotFoundError(
        "Place 'letter-recognition.data' in the notebook folder."
    )

letter_columns = [
    "letter",
    "x-box", "y-box", "width", "high", "onpix",
    "x-bar", "y-bar", "x2bar", "y2bar", "xybar",
    "x2ybr", "xy2br", "x-ege", "xegvy",
    "y-ege", "yegvx"
]

letter_df = pd.read_csv(
    LETTER_FILE,
    header=None,
    names=letter_columns
)

print(letter_df.head())
print("\nShape:", letter_df.shape)
print("\nClass count:", letter_df["letter"].nunique())


In [ ]:

# 10. Encode Letter Recognition labels A-Z as 0-25

classes = sorted(letter_df["letter"].unique())
class_to_int = {letter: i for i, letter in enumerate(classes)}

y_letter = letter_df["letter"].map(class_to_int).values.astype(int)

X_letter = letter_df.drop(columns=["letter"]).values.astype(np.float32)

# Standardize numerical features
X_letter = (
    X_letter - X_letter.mean(axis=0, keepdims=True)
) / (
    X_letter.std(axis=0, keepdims=True) + 1e-8
)

X_letter_train, X_letter_val, y_letter_train, y_letter_val = train_test_split(
    X_letter,
    y_letter,
    test_size=0.20,
    random_state=42,
    stratify=y_letter
)

print("Features:", X_letter.shape)
print("Training:", X_letter_train.shape)
print("Validation:", X_letter_val.shape)


## 11. Letter Recognition — data analysis and visualization

In [ ]:

# Class distribution

plt.figure(figsize=(12, 5))
letter_df["letter"].value_counts().sort_index().plot(kind="bar")
plt.xlabel("Letter")
plt.ylabel("Number of samples")
plt.title("Letter Recognition Class Distribution")
plt.grid(axis="y")
plt.show()


In [ ]:

# Feature histograms

letter_df.drop(columns=["letter"]).hist(
    figsize=(15, 12),
    bins=20
)

plt.suptitle("Feature Histograms")
plt.tight_layout()
plt.show()


In [ ]:

# Correlation heatmap

plt.figure(figsize=(12, 9))
sns.heatmap(
    letter_df.drop(columns=["letter"]).corr(),
    cmap="coolwarm",
    center=0
)

plt.title("Feature Correlation Heatmap")
plt.show()


## 12. Letter Recognition — activation function experiments

In [ ]:

activation_results = {}

for activation in ["relu", "tanh", "sigmoid"]:
    result = run_experiment(
        X_letter_train, y_letter_train,
        X_letter_val, y_letter_val,
        name=f"Letter Recognition — {activation.upper()}",
        hidden_layers=(128, 64),
        activation_name=activation,
        init_method="kaiming",
        epochs=50,
        learning_rate=0.1,
        batch_size=128,
        seed=42
    )

    activation_results[activation] = result


In [ ]:

activation_comparison = pd.DataFrame({
    "Activation": list(activation_results.keys()),
    "Validation Accuracy": [
        result["accuracy"] for result in activation_results.values()
    ]
})

activation_comparison


## 13. Letter Recognition — initialization experiments

In [ ]:

initialization_results = {}

for init in ["random", "xavier", "kaiming"]:
    result = run_experiment(
        X_letter_train, y_letter_train,
        X_letter_val, y_letter_val,
        name=f"Letter Recognition — {init.capitalize()} Initialization",
        hidden_layers=(128, 64),
        activation_name="relu",
        init_method=init,
        epochs=50,
        learning_rate=0.1,
        batch_size=128,
        seed=42
    )

    initialization_results[init] = result


In [ ]:

initialization_comparison = pd.DataFrame({
    "Initialization": list(initialization_results.keys()),
    "Validation Accuracy": [
        result["accuracy"] for result in initialization_results.values()
    ]
})

initialization_comparison


## 14. Letter Recognition — dropout regularization

In [ ]:

dropout_result = run_experiment(
    X_letter_train, y_letter_train,
    X_letter_val, y_letter_val,
    name="Letter Recognition — Dropout Regularization",
    hidden_layers=(128, 64),
    activation_name="relu",
    init_method="kaiming",
    dropout_rate=0.5,
    epochs=50,
    learning_rate=0.1,
    batch_size=128,
    seed=42
)


## 15. Letter Recognition — 2, 5 and 10 hidden layers

In [ ]:

architecture_results = {}

architectures = {
    "2 hidden layers": (128, 64),
    "5 hidden layers": (128, 128, 96, 64, 32),
    "10 hidden layers": (128, 128, 128, 128, 96, 96, 64, 64, 32, 32)
}

for name, architecture in architectures.items():
    result = run_experiment(
        X_letter_train, y_letter_train,
        X_letter_val, y_letter_val,
        name=f"Letter Recognition — {name}",
        hidden_layers=architecture,
        activation_name="relu",
        init_method="kaiming",
        epochs=50,
        learning_rate=0.1,
        batch_size=128,
        seed=42
    )

    architecture_results[name] = result


In [ ]:

architecture_comparison = pd.DataFrame({
    "Architecture": list(architecture_results.keys()),
    "Validation Accuracy": [
        result["accuracy"] for result in architecture_results.values()
    ]
})

architecture_comparison


## 16. Confusion matrices

In [ ]:

# Confusion matrix for the Kaiming/ReLU model

kaiming_letter_pred = initialization_results["kaiming"]["predictions"]

show_confusion_matrix(
    y_letter_val,
    kaiming_letter_pred,
    class_names=classes,
    title="Letter Recognition — Kaiming Initialization"
)


In [ ]:

# Confusion matrix for the 5-hidden-layer model

pred_5 = architecture_results["5 hidden layers"]["predictions"]

show_confusion_matrix(
    y_letter_val,
    pred_5,
    class_names=classes,
    title="Letter Recognition — 5 Hidden Layers"
)


In [ ]:

# Confusion matrix for the 10-hidden-layer model

pred_10 = architecture_results["10 hidden layers"]["predictions"]

show_confusion_matrix(
    y_letter_val,
    pred_10,
    class_names=classes,
    title="Letter Recognition — 10 Hidden Layers"
)
